In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nqa112/vizwiz-2023-edition")

print("Path to dataset files:", path)

100%|██████████| 17.5G/17.5G [03:01<00:00, 103MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/nqa112/vizwiz-2023-edition/versions/1


In [4]:
!ls /root/.cache/kagglehub/datasets/nqa112/vizwiz-2023-edition/versions/1/Annotations/val.json

/root/.cache/kagglehub/datasets/nqa112/vizwiz-2023-edition/versions/1/Annotations/val.json


# BLUE-1 Score

In [6]:
import json
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Đọc dữ liệu
with open('/content/default_pred.json', 'r') as f:
    predictions = json.load(f)

with open('/root/.cache/kagglehub/datasets/nqa112/vizwiz-2023-edition/versions/1/Annotations/val.json', 'r') as f:
    ground_truths = json.load(f)

# Tạo dictionary cho ground truth để dễ tra cứu
gt_dict = {item['image']: item['answers'] for item in ground_truths}

# Khởi tạo
scores = []
smoothie = SmoothingFunction().method4

for pred in predictions:
    image_id = pred['image']
    pred_answer = pred['answer']

    # Tokenize
    pred_tokens = pred_answer.lower().split()

    # Lấy các ground truth answers
    references = [ans['answer'].lower().split() for ans in gt_dict.get(image_id, []) if 'answer' in ans]

    # Tính BLEU cho từng câu
    if references:
        bleu = sentence_bleu(references, pred_tokens, smoothing_function=smoothie)
        scores.append(bleu)

# Tính trung bình BLEU score
average_bleu = sum(scores) / len(scores) if scores else 0.0
print(f"Average BLEU score: {average_bleu:.4f}")

Average BLEU score: 0.6981


# VQA Accuracy

In [7]:
import json
from collections import Counter

# Đọc dữ liệu
with open('/content/default_pred.json') as f:
    predictions = json.load(f)

with open('/root/.cache/kagglehub/datasets/nqa112/vizwiz-2023-edition/versions/1/Annotations/val.json') as f:
    ground_truths = json.load(f)

# Tạo dict tra cứu ground truth theo image_id
gt_dict = {item['image']: item['answers'] for item in ground_truths}

accuracies = []

for pred in predictions:
    image_id = pred['image']
    pred_answer = pred['answer'].strip().lower()

    # Lấy các câu trả lời ground truth (chuyển về chữ thường và strip)
    gt_answers = [
        ans['answer'].strip().lower()
        for ans in gt_dict.get(image_id, [])
        if 'answer' in ans
    ]

    # Đếm số lần dự đoán trùng với các câu trả lời
    matching_count = sum(1 for ans in gt_answers if ans == pred_answer)

    # Tính accuracy chuẩn VizWiz
    acc = min(matching_count / 3, 1.0)
    accuracies.append(acc)

# Tính trung bình
average_accuracy = sum(accuracies) / len(accuracies) if accuracies else 0.0
print(f"VizWiz-style VQA Accuracy: {average_accuracy:.4f}")

VizWiz-style VQA Accuracy: 0.6132


# WUPS

In [ ]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')

In [ ]:
from nltk.corpus import wordnet as wn

def wup_similarity(word1, word2):
    synsets1 = wn.synsets(word1)
    synsets2 = wn.synsets(word2)

    if not synsets1 or not synsets2:
        return 0.0  # Không tìm được nghĩa -> 0 điểm

    max_score = 0.0
    for syn1 in synsets1:
        for syn2 in synsets2:
            score = syn1.wup_similarity(syn2)
            if score is not None:
                max_score = max(max_score, score)
    return max_score

# WUPS Hard-match (0.9)

In [9]:
import json

# Đọc dữ liệu
with open('/content/default_pred.json') as f:
    predictions = json.load(f)

with open('/root/.cache/kagglehub/datasets/nqa112/vizwiz-2023-edition/versions/1/Annotations/val.json') as f:
    ground_truths = json.load(f)

# Tạo dict tra cứu
gt_dict = {item['image']: item['answers'] for item in ground_truths}

def compute_wups(pred_answer, gt_answers, threshold=0.9):
    scores = []
    for ans in gt_answers:
        gt = ans['answer'].strip().lower()
        pred = pred_answer.strip().lower()
        sim = wup_similarity(pred, gt)
        if sim >= threshold:
            scores.append(1.0)
        else:
            scores.append(sim)
    return max(scores) if scores else 0.0

# Tính điểm WUPS trung bình
scores = []
for pred in predictions:
    image_id = pred['image']
    pred_answer = pred['answer']
    gt_answers = gt_dict.get(image_id, [])
    score = compute_wups(pred_answer, gt_answers)
    scores.append(score)

avg_wups = sum(scores) / len(scores) if scores else 0.0
print(f"Average WUPS Score: {avg_wups:.4f}")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Average WUPS Score: 0.7638


# WUPS Soft-match (0.0)

In [12]:
import json

# Đọc dữ liệu
with open('/content/default_pred.json') as f:
    predictions = json.load(f)

with open('/root/.cache/kagglehub/datasets/nqa112/vizwiz-2023-edition/versions/1/Annotations/val.json') as f:
    ground_truths = json.load(f)

# Tạo dict tra cứu
gt_dict = {item['image']: item['answers'] for item in ground_truths}

def compute_wups(pred_answer, gt_answers, threshold=0.0):
    scores = []
    for ans in gt_answers:
        gt = ans['answer'].strip().lower()
        pred = pred_answer.strip().lower()
        sim = wup_similarity(pred, gt)
        if sim >= threshold:
            scores.append(1.0)
        else:
            scores.append(sim)
    return max(scores) if scores else 0.0

# Tính điểm WUPS trung bình
scores = []
for pred in predictions:
    image_id = pred['image']
    pred_answer = pred['answer']
    gt_answers = gt_dict.get(image_id, [])
    score = compute_wups(pred_answer, gt_answers)
    scores.append(score)

avg_wups = sum(scores) / len(scores) if scores else 0.0
print(f"Average WUPS Score: {avg_wups:.4f}")

Average WUPS Score: 1.0000
